# 01 — Data Cleaning

**Project:** Food Delivery Operations Analytics  
**Author:** Sitanshu Singh  
**Date:** Jan 2024

This notebook loads the raw datasets, merges them, and handles all the data quality issues before analysis.
Run this first before any of the EDA notebooks.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

print('Libraries loaded')

## 1. Load Raw Data

In [ ]:
# Load the raw CSVs
orders_raw = pd.read_csv('../data/raw/orders_raw.csv')
restaurants_raw = pd.read_csv('../data/raw/restaurants_raw.csv')
customers_raw = pd.read_csv('../data/raw/customers_raw.csv')
riders_raw = pd.read_csv('../data/raw/delivery_partners_raw.csv')

print(f'Orders: {orders_raw.shape}')
print(f'Restaurants: {restaurants_raw.shape}')
print(f'Customers: {customers_raw.shape}')
print(f'Riders: {riders_raw.shape}')

## 2. Check for Missing Values

In [ ]:
print('Missing values in orders:')
print(orders_raw.isnull().sum())
print()
print('Missing values in restaurants:')
print(restaurants_raw.isnull().sum())

## 3. Fix City Name Inconsistencies

In [ ]:
# There were 14 variants for city names in the raw data
city_mapping = {
    'Bengaluru': 'Bangalore',
    'Bangalore, KA': 'Bangalore',
    'blr': 'Bangalore',
    'bombay': 'Mumbai',
    'Mumbai, MH': 'Mumbai',
    'new delhi': 'Delhi',
    'Delhi NCR': 'Delhi',
    'Hyd': 'Hyderabad',
    'Kolkatta': 'Kolkata',
    'kolkata': 'Kolkata',
    'ahmedabad': 'Ahmedabad',
    'pune': 'Pune',
    'Chennai, TN': 'Chennai',
}

customers_raw['city'] = customers_raw['city'].replace(city_mapping)
restaurants_raw['city'] = restaurants_raw['city'].replace(city_mapping)

print('Unique cities after cleaning:', sorted(customers_raw['city'].unique()))

## 4. Fix Missing Delivery Times

In [ ]:
# Fill missing delivery_time_mins with city + restaurant median
# First merge to get city info
orders_with_city = orders_raw.merge(
    customers_raw[['customer_id', 'city']],
    on='customer_id', how='left'
)

city_rest_median = (
    orders_with_city
    .groupby(['city', 'restaurant_id'])['delivery_time_mins']
    .median()
    .reset_index()
    .rename(columns={'delivery_time_mins': 'median_time'})
)

orders_with_city = orders_with_city.merge(city_rest_median, on=['city', 'restaurant_id'], how='left')
orders_with_city['delivery_time_mins'] = orders_with_city['delivery_time_mins'].fillna(
    orders_with_city['median_time']
)

print(f'Nulls remaining: {orders_with_city["delivery_time_mins"].isnull().sum()}')

## 5. Remove Duplicates and Bad Records

In [ ]:
# Drop duplicate rows
before = len(orders_with_city)
orders_clean = orders_with_city.drop_duplicates()
print(f'Removed {before - len(orders_clean)} duplicates')

# Drop rows with negative delivery time
negative_mask = orders_clean['delivery_time_mins'] < 0
print(f'Dropping {negative_mask.sum()} rows with negative delivery time')
orders_clean = orders_clean[~negative_mask]

# Flag outlier orders (> 5000 INR) — don't drop, just flag
orders_clean['is_outlier'] = orders_clean['order_value'] > 5000
print(f'Outlier orders flagged: {orders_clean["is_outlier"].sum()}')

# Set ratings of 0.0 to NaN
orders_clean['rating'] = orders_clean['rating'].replace(0.0, np.nan)

print(f'\nFinal order count: {len(orders_clean)}')

## 6. Standardize Timestamps

In [ ]:
# The raw data had several timestamp formats mixed together
orders_clean['order_date'] = pd.to_datetime(orders_clean['order_date'], infer_datetime_format=True)

# Extract useful time features
orders_clean['order_hour'] = orders_clean['order_date'].dt.hour
orders_clean['order_dayofweek'] = orders_clean['order_date'].dt.dayofweek  # 0=Monday
orders_clean['order_month'] = orders_clean['order_date'].dt.month
orders_clean['is_weekend'] = orders_clean['order_dayofweek'].isin([5, 6])

print('Date range:', orders_clean['order_date'].min(), 'to', orders_clean['order_date'].max())

## 7. Export Cleaned Data

In [ ]:
orders_clean.to_csv('../data/cleaned/orders_cleaned.csv', index=False)
customers_raw.to_csv('../data/cleaned/customers_cleaned.csv', index=False)
restaurants_raw.to_csv('../data/cleaned/restaurants_cleaned.csv', index=False)

print('Done — cleaned files saved to data/cleaned/')
print(f'Final shape: {orders_clean.shape}')